# Explore an entity type

Inspect the properties CluedIn holds for an entity type, so that you know
which vocabulary keys are worth adding to the metrics configuration.


In [ ]:
# The notebooks run from the repository root so that the modules import.
import os
import sys
import warnings

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# Every input comes from the metrics configuration -- no hard-coded entity
# types or vocabulary keys. Point METRICS_CONFIG at your own file.
CONFIG_PATH = os.getenv('METRICS_CONFIG', 'config/metrics.example.json')
CONFIG_PATH


In [ ]:
%pip install matplotlib


In [ ]:
from config import load_config

config = load_config(CONFIG_PATH)
[e.entity_type for e in config.entity_types]


## Load one entity type

Change `ENTITY_TYPE` to any type in your configuration.


In [ ]:
from api import get_entities_by_type

ENTITY_TYPE = config.entity_types[0].entity_type

entities = get_entities_by_type(ENTITY_TYPE, page_size=config.page_size)
print(f'{len(entities)} {ENTITY_TYPE}')


## Which properties exist, and how often are they populated?


In [ ]:
from collections import Counter

populated = Counter()
for entity in entities:
    populated.update(entity['properties'].keys())

total = len(entities) or 1
for name, count in populated.most_common(40):
    print(f'{count / total:6.1%}  {count:>7}  {name}')


## Plot two numeric properties

For geographic data, set these to your latitude and longitude keys.


In [ ]:
import matplotlib.pyplot as plt

from rules import cast_to_float, get_property

X_KEY = None  # e.g. 'example.location.longitude'
Y_KEY = None  # e.g. 'example.location.latitude'

if X_KEY and Y_KEY:
    points = [
        (cast_to_float(get_property(e, X_KEY)),
         cast_to_float(get_property(e, Y_KEY)))
        for e in entities
    ]
    points = [(x, y) for x, y in points if x is not None and y is not None]
    plt.figure(figsize=(12, 6))
    plt.scatter([x for x, _ in points], [y for _, y in points],
                s=0.5, alpha=0.35)
    plt.xlabel(X_KEY)
    plt.ylabel(Y_KEY)
    plt.title(f'{ENTITY_TYPE}: {len(points)} points')
    plt.show()
else:
    print('Set X_KEY and Y_KEY to plot.')
